In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import HDBSCAN
import shap
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [ ]:
# 1. 데이터 로드 및 전처리
df = pd.read_csv("water_quality_data.csv", encoding='euc-kr')
features = df.drop(columns=['spot']).values

In [ ]:
# 2. [추가] Iterative Imputer 도입
# 다른 수질 항목들과의 상관관계를 회귀 모델로 학습하여 결측치를 예측해서 채웁니다.
imputer = IterativeImputer(max_iter=10, random_state=0)
features_imputed = imputer.fit_transform(features)

In [ ]:
# 3. 데이터 표준화 및 VAE 입력
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
X_tensor = torch.FloatTensor(X_scaled)

In [ ]:
# 4. [AI 기법 1] VAE (Variational Autoencoder) - 데이터 현미경
# 데이터의 복잡한 관계를 압축하여 핵심 특징(Latent space)만 추출합니다.
class VAE(nn.Module):
    def __init__(self, input_dim):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 2) # 2차원 잠재 공간으로 압축
        )
        self.decoder = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim)
        )
        
    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return reconstructed, z

model = VAE(features.shape[1])
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 모델 학습 (간략화)
for epoch in range(100):
    recon, z = model(X_tensor)
    loss = nn.MSELoss()(recon, X_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
# 5. [AI 기법 2] HDBSCAN - 밀도 기반 군집화
# 정답 없이도 데이터가 뭉쳐 있는 '밀도'를 파악해 자동으로 오염 그룹을 생성합니다.
_, latent_data = model(X_tensor)
latent_data = latent_data.detach().numpy()
clusterer = HDBSCAN(min_cluster_size=5)
df['Cluster'] = clusterer.fit_predict(latent_data)

In [ ]:
# 6. [AI 기법 3] SHAP - 판단 근거 설명 (XAI)
# 특정 시료가 왜 해당 그룹으로 분류되었는지 '이유'를 설명합니다.
def model_predict(data):
    _, z = model(torch.FloatTensor(data))
    return z.detach().numpy()

# SHAP Explainer (KernelSHAP 사용)
explainer = shap.KernelExplainer(lambda x: model_predict(x)[:, 0], X_scaled[:100])
shap_values = explainer.shap_values(X_scaled[0:1])

# 결과 출력 (첫 번째 시료에 대한 분석 예시)
print(f"시료 [0]의 판단 근거:\n{shap_values}")
shap.summary_plot(shap_values, X_scaled, feature_names=df.columns[1:-1])